<a href="https://colab.research.google.com/github/CamiloR11/PROGCOM_B_2026/blob/main/GQ3_PROBLEMA_ASOCIADO_A_LA_CARRERA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Planificador Diario de Tareas Empresariales - Optimizado
# Propósito: Organizar tareas del día según urgencia y duración, dividiendo tareas largas automáticamente
# Jornada: 7:00-17:00

from typing import List, Tuple

class Tarea:
    def __init__(self, nombre: str, duracion: int, urgencia: int):
        self.nombre = nombre
        self.duracion = duracion
        self.urgencia = urgencia  # 1=alta,2=media,3=baja

    def __repr__(self):
        return f"{self.nombre} ({self.duracion}h, urgencia={self.urgencia})"

class PlanificadorDiario:
    def __init__(self):
        self.jornada_inicio = 7
        self.jornada_fin = 17
        self.almuerzo_inicio = 12
        self.almuerzo_fin = 14

    def organizar_tareas(self, tareas: List[Tarea]) -> Tuple[List[Tuple[str, float, float]], List[Tarea]]:
        """Organiza las tareas del día, dividiendo tareas largas y devolviendo horario y pendientes"""
        tareas_ordenadas = sorted(tareas, key=lambda t: t.urgencia)
        hora_actual = self.jornada_inicio
        agenda = []
        pendientes = []

        for tarea in tareas_ordenadas:
            horas_restantes = tarea.duracion

            while horas_restantes > 0:
                if self.almuerzo_inicio <= hora_actual < self.almuerzo_fin:
                    hora_actual = self.almuerzo_fin

                if hora_actual < self.almuerzo_inicio:
                    disponible = self.almuerzo_inicio - hora_actual
                else:
                    disponible = self.jornada_fin - hora_actual

                if disponible <= 0:
                    if horas_restantes > 0:
                        pendientes.append(Tarea(tarea.nombre, horas_restantes, tarea.urgencia))
                    break

                horas_a_programar = min(horas_restantes, disponible)
                inicio = hora_actual
                fin = hora_actual + horas_a_programar
                agenda.append((tarea.nombre, inicio, fin))
                horas_restantes -= horas_a_programar
                hora_actual = fin

                if hora_actual >= self.jornada_fin and horas_restantes > 0:
                    pendientes.append(Tarea(tarea.nombre, horas_restantes, tarea.urgencia))
                    break

        return agenda, pendientes

    def imprimir_agenda(self, agenda: List[Tuple[str, float, float]]):
        print("\n--- AGENDA DEL DÍA ---")
        for tarea, inicio, fin in agenda:
            print(f"{int(inicio)}:00 - {int(fin)}:00 | {tarea}")

    def imprimir_pendientes(self, pendientes: List[Tarea]):
        if pendientes:
            print("\n--- ACTIVIDADES PENDIENTES PARA EL DÍA SIGUIENTE ---")
            for tarea in pendientes:
                print(f"{tarea.nombre} ({tarea.duracion}h, urgencia={tarea.urgencia})")
        else:
            print("\nNo quedan actividades pendientes para el día siguiente.")


def main():
    planificador = PlanificadorDiario()
    tareas = []

    print("Ingrese las tareas del día. Escriba 'fin' como nombre para terminar.\n")
    while True:
        nombre = input("Nombre de la tarea: ")
        if nombre.lower() == 'fin':
            break
        try:
            duracion = int(input("Duración en horas: "))
            urgencia = int(input("Grado de importancia (1=alta,2=media,3=baja): "))
            tareas.append(Tarea(nombre, duracion, urgencia))
            print("Tarea agregada.\n")
        except ValueError:
            print("Entrada inválida, intente nuevamente.\n")

    agenda, pendientes = planificador.organizar_tareas(tareas)
    planificador.imprimir_agenda(agenda)
    planificador.imprimir_pendientes(pendientes)


if __name__ == "__main__":
    main()

In [1]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import json


class Tarea:
    def __init__(self, nombre, duracion, urgencia):
        self.nombre = nombre
        self.duracion = duracion
        self.urgencia = urgencia

    def to_dict(self):
        return {
            "nombre": self.nombre,
            "duracion": self.duracion,
            "urgencia": self.urgencia
        }


class PlanificadorDiario:
    def __init__(self):
        self.jornada_inicio = 7
        self.jornada_fin = 17
        self.almuerzo_inicio = 12
        self.almuerzo_fin = 14

    def organizar_tareas(self, tareas):
        tareas_ordenadas = sorted(tareas, key=lambda t: t.urgencia)

        hora_actual = self.jornada_inicio
        agenda = []
        pendientes = []

        for tarea in tareas_ordenadas:
            horas_restantes = tarea.duracion

            while horas_restantes > 0:

                if self.almuerzo_inicio <= hora_actual < self.almuerzo_fin:
                    hora_actual = self.almuerzo_fin

                if hora_actual < self.almuerzo_inicio:
                    disponible = self.almuerzo_inicio - hora_actual
                else:
                    disponible = self.jornada_fin - hora_actual

                if disponible <= 0:
                    pendientes.append(
                        Tarea(
                            tarea.nombre,
                            horas_restantes,
                            tarea.urgencia
                        )
                    )
                    break

                horas_a_programar = min(
                    horas_restantes,
                    disponible
                )

                inicio = hora_actual
                fin = hora_actual + horas_a_programar

                agenda.append(
                    (tarea.nombre, inicio, fin)
                )

                horas_restantes -= horas_a_programar
                hora_actual = fin

                if hora_actual >= self.jornada_fin:
                    if horas_restantes > 0:
                        pendientes.append(
                            Tarea(
                                tarea.nombre,
                                horas_restantes,
                                tarea.urgencia
                            )
                        )
                    break

        return agenda, pendientes


class Aplicacion:

    def __init__(self, root):

        self.root = root
        self.root.title("Planificador Diario Empresarial")
        self.root.geometry("900x600")

        self.planificador = PlanificadorDiario()
        self.tareas = []

        # FRAME FORMULARIO
        frame = tk.Frame(root)
        frame.pack(pady=10)

        tk.Label(frame, text="Nombre").grid(row=0, column=0)
        tk.Label(frame, text="Duración").grid(row=0, column=1)
        tk.Label(frame, text="Urgencia").grid(row=0, column=2)

        self.entry_nombre = tk.Entry(frame)
        self.entry_nombre.grid(row=1, column=0)

        self.entry_duracion = tk.Entry(frame)
        self.entry_duracion.grid(row=1, column=1)

        self.combo_urgencia = ttk.Combobox(
            frame,
            values=[1, 2, 3],
            width=5
        )
        self.combo_urgencia.grid(row=1, column=2)
        self.combo_urgencia.current(0)

        tk.Button(
            frame,
            text="Agregar tarea",
            command=self.agregar_tarea
        ).grid(row=1, column=3, padx=10)

        # LISTA TAREAS
        self.lista_tareas = tk.Listbox(root, width=80, height=10)
        self.lista_tareas.pack(pady=10)

        # BOTONES
        botones = tk.Frame(root)
        botones.pack()

        tk.Button(
            botones,
            text="Generar Agenda",
            command=self.generar_agenda
        ).grid(row=0, column=0, padx=5)

        tk.Button(
            botones,
            text="Guardar Tareas",
            command=self.guardar_archivo
        ).grid(row=0, column=1, padx=5)

        tk.Button(
            botones,
            text="Abrir Archivo",
            command=self.abrir_archivo
        ).grid(row=0, column=2, padx=5)

        tk.Button(
            botones,
            text="Exportar Agenda",
            command=self.exportar_agenda
        ).grid(row=0, column=3, padx=5)

        # RESULTADO
        self.texto_resultado = tk.Text(root, width=100, height=20)
        self.texto_resultado.pack(pady=10)

    def agregar_tarea(self):

        try:
            nombre = self.entry_nombre.get()
            duracion = int(self.entry_duracion.get())
            urgencia = int(self.combo_urgencia.get())

            tarea = Tarea(
                nombre,
                duracion,
                urgencia
            )

            self.tareas.append(tarea)

            self.lista_tareas.insert(
                tk.END,
                f"{nombre} - {duracion}h - urgencia {urgencia}"
            )

            self.entry_nombre.delete(0, tk.END)
            self.entry_duracion.delete(0, tk.END)

        except:
            messagebox.showerror(
                "Error",
                "Datos inválidos"
            )

    def generar_agenda(self):

        agenda, pendientes = self.planificador.organizar_tareas(
            self.tareas
        )

        self.ultima_agenda = agenda
        self.ultimo_pendiente = pendientes

        self.texto_resultado.delete(1.0, tk.END)

        self.texto_resultado.insert(
            tk.END,
            "=== AGENDA DEL DÍA ===\n\n"
        )

        for tarea, inicio, fin in agenda:
            self.texto_resultado.insert(
                tk.END,
                f"{int(inicio)}:00 - {int(fin)}:00 | {tarea}\n"
            )

        self.texto_resultado.insert(
            tk.END,
            "\n=== PENDIENTES ===\n\n"
        )

        if pendientes:
            for p in pendientes:
                self.texto_resultado.insert(
                    tk.END,
                    f"{p.nombre} ({p.duracion}h)\n"
                )
        else:
            self.texto_resultado.insert(
                tk.END,
                "No hay pendientes\n"
            )

    def guardar_archivo(self):

        archivo = filedialog.asksaveasfilename(
            defaultextension=".json"
        )

        if archivo:

            datos = [
                t.to_dict()
                for t in self.tareas
            ]

            with open(archivo, "w") as f:
                json.dump(datos, f)

            messagebox.showinfo(
                "Guardado",
                "Archivo guardado correctamente"
            )

    def abrir_archivo(self):

        archivo = filedialog.askopenfilename(
            filetypes=[("JSON", "*.json")]
        )

        if archivo:

            with open(archivo, "r") as f:
                datos = json.load(f)

            self.tareas.clear()
            self.lista_tareas.delete(0, tk.END)

            for d in datos:

                tarea = Tarea(
                    d["nombre"],
                    d["duracion"],
                    d["urgencia"]
                )

                self.tareas.append(tarea)

                self.lista_tareas.insert(
                    tk.END,
                    f"{tarea.nombre} - {tarea.duracion}h - urgencia {tarea.urgencia}"
                )

            messagebox.showinfo(
                "Carga",
                "Archivo cargado correctamente"
            )

    def exportar_agenda(self):

        archivo = filedialog.asksaveasfilename(
            defaultextension=".txt"
        )

        if archivo:

            with open(archivo, "w") as f:

                f.write("=== AGENDA ===\n\n")

                for tarea, inicio, fin in self.ultima_agenda:
                    f.write(
                        f"{int(inicio)}:00 - {int(fin)}:00 | {tarea}\n"
                    )

                f.write("\n=== PENDIENTES ===\n\n")

                for p in self.ultimo_pendiente:
                    f.write(
                        f"{p.nombre} ({p.duracion}h)\n"
                    )

            messagebox.showinfo(
                "Exportado",
                "Agenda exportada"
            )


root = tk.Tk()
app = Aplicacion(root)
root.mainloop()

TclError: no display name and no $DISPLAY environment variable